# **DATA FEATURE ENGINEERING**

In [ ]:
import sys
sys.path.append('/projects/ngs_eco/users/kmvr819/PODS/GeMinAI/samecode/')
sys.path.append('/wscratch/ai_data_center/ods_eds_aidc_hub/')

**Explanation:**

This cell adds two directories to the system path, allowing you to import modules from these directories in the notebook.

**Purpose:**

Adds directories to **sys.path** so Python can find and import modules from these custom paths.

In [ ]:
from aidc_hub import load_dataset
from aidc_hub import list_catalog
import numpy as np
import pandas as pd

**from aidc_hub import load_dataset, list_catalog:**

These are likely custom functions for loading datasets and listing available datasets or catalog information.

**import numpy as np:**

Imports the **numpy** library for numerical operations.

**import pandas as pd:**

Imports the **pandas** library for data manipulation and analysis.

**Explanation:**

**load_dataset and list_catalog** are being imported from a module named **aidc_hub.** This module likely exists in one of the directories you added to the system path in Cell 1.

**numpy (imported as np) and pandas (imported as pd)** are standard Python libraries used for numerical computations and data manipulation, respectively.

In [ ]:
from samecode.signatures import median_aggregation_df
from samecode.plot.pyplot import subplots

**from samecode.signatures import median_aggregation_df:**

Imports a function for aggregating data using the median, possibly used for summarizing or processing data.

**from samecode.plot.pyplot import subplots:**

Imports a custom subplots function for creating multiple plots in a single figure.

**Explanation:**

**median_aggregation_df** is imported from **samecode.signatures.** This function might be related to data aggregation or statistical operations.

**subplots** is imported from **samecode.plot.pyplot**, likely a custom plotting function.

# **Harmonize TCGA and GTEx data for foundation model**


**load signatures**

We are using bagaev et al + immune related signatures.

In [ ]:
# Load signatures
signatures = pd.read_csv('../../../../../../../../signatures/immune_cell_signatures/results/fges+io.gencode.v23.csv')

**Process TCGA**

In [ ]:
# get TCGA studies
studies = [i for i in list_catalog('cbioportal') if 'pan_can_atlas_2018' in i]
len(studies)

1. **list_catalog('cbioportal'):**

**Function Call: **The function **list_catalog** is called with the argument **'cbioportal'.**

**Purpose:**

This function is likely designed to return a list of available studies or datasets from the cBioPortal, which is a popular resource for exploring cancer genomics data.

The argument **'cbioportal'** specifies the source or catalog from which to retrieve the list.

**2. List Comprehension: [i for i in list_catalog('cbioportal') if 'pan_can_atlas_2018' in i]**

**Explanation:**

This is a Python list comprehension that creates a new list by iterating over each item i in the list returned by **list_catalog('cbioportal').**

**Filter Condition: if 'pan_can_atlas_2018' in i** filters the items, including only those that contain the string **'pan_can_atlas_2018'.**

The result is a list called **studies** that contains only those studies related to the Pan-Cancer Atlas 2018 project, a comprehensive analysis of multiple cancer types.

**len(studies):**

**Function Call: **The **len** function is called on the **studies** list.

**Purpose:**

This returns the number of studies in the filtered list, i.e., the total count of studies related to **'pan_can_atlas_2018'** found in the cBioPortal catalog.

**Summary**

The code retrieves a list of studies from the cBioPortal.

It filters this list to include only those studies related to the "Pan-Cancer Atlas 2018" project.

Finally, it calculates and returns the number of such studies.

If you run this code, you will get the number of studies in the cBioPortal catalog that are part of the Pan-Cancer Atlas 2018 project.

In [ ]:
# rnaseq_profile
sig_data = []
for study in studies:
    dataset = load_dataset('cbioportal/{}'.format(study))
    rna = dataset.modality['rnaseq']
    rna.to_ensembl()
    rna.transform(np.log2, add=0.001)
    table = rna.data.reset_index(drop=True)

    sgs = median_aggregation_df(table, signatures=signatures, signature_column='signature', gene_column='gene_id', median_scaling=True)
    sgs['patient_id'] = table['patient_id']
    sgs['tissue'] = dataset.modality['clinical'].data[['CLIN_CANCER_TYPE']].values[0][0]
    sgs['source_data'] = 'TCGA'

    sig_data.append(sgs)

The error you're encountering **(ImportError: Missing optional dependency 'pyarrow')** indicates that the **pyarrow** library is not installed in your environment. This library is necessary for reading and writing data in the Feather format, which is likely the format of the data you're working with.

**Step 1: Install pyarrow**
You need to install the pyarrow library. In your JupyterLab, you can install it by running the following command in a new cell:

**!pip install pyarrow**


**Step 2: Rerun Your Code**

Once **pyarrow **is installed, you can rerun your code. Here's a breakdown of what your code does and what changes might be needed for better clarity:

In [ ]:
# Initialize an empty list to store signature data for each study
sig_data = []

# Loop through each study in the 'studies' list
for study in studies:
    # Load the dataset for the current study from cBioPortal
    dataset = load_dataset('cbioportal/{}'.format(study))

    # Access the RNA-seq data from the dataset
    rna = dataset.modality['rnaseq']

    # Convert gene identifiers to Ensembl format (common in RNA-seq data)
    rna.to_ensembl()

    # Apply a log2 transformation to the RNA-seq data to normalize it
    # Adding a small constant (0.001) to avoid taking the log of zero
    rna.transform(np.log2, add=0.001)

    # Convert the transformed RNA-seq data into a pandas DataFrame
    table = rna.data.reset_index(drop=True)

    # Perform median aggregation on the data using the provided signatures
    sgs = median_aggregation_df(
        table,
        signatures=signatures,
        signature_column='signature',
        gene_column='gene_id',
        median_scaling=True
    )

    # Add patient IDs to the aggregated data
    sgs['patient_id'] = table['patient_id']

    # Add tissue type information from the clinical data associated with the dataset
    sgs['tissue'] = dataset.modality['clinical'].data[['CLIN_CANCER_TYPE']].values[0][0]

    # Label the source of the data as 'TCGA' (The Cancer Genome Atlas)
    sgs['source_data'] = 'TCGA'

    # Append the processed data for this study to the sig_data list
    sig_data.append(sgs)


**Explanation of Each Code Line:**

**sig_data = []:** Initializes an empty list that will store the signature data for each study.

**for study in studies:** Iterates over each study in the list **studies.**

**dataset = load_dataset('cbioportal/{}'.format(study)):** Loads the dataset for the current study from the cBioPortal using the **load_dataset** function.

**rna = dataset.modality['rnaseq']: **Extracts the RNA-seq data modality from the dataset.

**rna.to_ensembl():** Converts gene identifiers in the RNA-seq data to Ensembl IDs, a common identifier format for genes.

**rna.transform(np.log2, add=0.001):** Applies a log2 transformation to the RNA-seq data to normalize it. A small constant (0.001) is added to avoid issues with zero values.

**table = rna.data.reset_index(drop=True): **Converts the transformed RNA-seq data to a DataFrame and resets the index.

**sgs = median_aggregation_df(...): **Aggregates the RNA-seq data by computing the median for each signature using the **median_aggregation_df **function. The function requires information about signatures, signature columns, gene columns, and whether to apply median scaling.

**sgs['patient_id'] = table['patient_id']: Adds a patient_id** column to the aggregated data.

**sgs['tissue'] = dataset.modality['clinical'].data[['CLIN_CANCER_TYPE']].values[0][0]:** Adds tissue type information to the aggregated data, extracted from the clinical data associated with the dataset.

**sgs['source_data'] = 'TCGA':** Labels the source of the data as 'TCGA'.

**sig_data.append(sgs):** Appends the processed signature data for the current study to the **sig_data **list.

**Step 3: Run the Code**

After installing pyarrow, rerun the code block to ensure everything works correctly.

**Process GTEx**

In [ ]:
# get GTEx studies
studies = list_catalog('gtex_v8')

In [ ]:
for study in studies:
    dataset = load_dataset('gtex_v8/{}'.format(study))
    rna = dataset.modality['rnaseq']
    rna.to_ensembl()
    rna.transform(np.log2, add=0.001)
    table = rna.data.reset_index(drop=True)

    sgs = median_aggregation_df(table, signatures=signatures, signature_column='signature', gene_column='gene_id', median_scaling=True)
    sgs['patient_id'] = table['patient_id']
    sgs['tissue'] = study
    sgs['source_data'] = 'GTEx'

    sig_data.append(sgs)

**Merge and save data**

In [ ]:
data = pd.concat(sig_data).reset_index(drop=True)

In [ ]:
data.head()

In [ ]:
data.shape

In [ ]:
data.to_csv('../data/gtex+tcga+FGEs+io.csv', index=False)


**1. data.to_csv(...):**

This method is called on a pandas DataFrame object, **data**.

The **to_csv** method is used to write the contents of the DataFrame to a CSV file. CSV is a common format for storing tabular data in plain text, where each line represents a row in the table, and columns are separated by commas.

**2. '../data/gtex+tcga+FGEs+io.csv':**

This is the file path where the CSV file will be saved.

**../** indicates that the file will be saved one directory up from the current working directory in a folder named **data**.

The file name is **gtex+tcga+FGEs+io.csv**, which suggests that the data being saved might involve multiple datasets, such as GTEx (Genotype-Tissue Expression), TCGA (The Cancer Genome Atlas), FGEs (possibly referring to functional genomics experiments), and IO (possibly indicating Immuno-Oncology).

**3. index=False:**

The **index** parameter controls whether the row indices of the DataFrame should be written to the CSV file.

**index=False** means that the row indices will not be saved in the CSV file. This is typically done when the index is not meaningful or if you want a cleaner file with only the data columns.

**Probable Outcome**

1. **File Creation:** This command will create a CSV file named gtex+tcga+FGEs+io.csv in the ../data/ directory relative to your current working directory.

**2. File Content:** The content of the CSV file will reflect the data present in the **data** DataFrame. Each row in the DataFrame will become a row in the CSV file, and each column will become a column in the CSV file.

**3. No Index:** The file will not include the row indices (usually the first column in a CSV if **index=True**), making the file look more like a typical spreadsheet without additional row numbers.

**Use Case**

This operation is useful when you want to save your processed or aggregated data to a file for later use, sharing, or analysis with other tools that can read CSV files, such as Excel, R, or another Python script.

If you navigate to the **../data/ **directory after running this code, you should find the **gtex+tcga+FGEs+io.csv **file with your data inside it.

**Visualizations**

In [ ]:
import seaborn as sns

In [ ]:
# signatures are normalized to the median
sns.boxplot(data=data, x='T cells', y='source_data')

In [ ]:
axs = subplots(rows=1, cols=1, w=5, h=12)
sns.boxplot(data=data, x='T cells', y='tissue', ax=axs[0])